# Housing costs from the datathon Zillow listings (alternative version): Albert
**Datathon for Social Good 2026 · Bay Area Affordability Project**

Finds the **typical rent and typical home price in every Bay Area zip code** from the Zillow listings, plus rent by bedroom count, price per square foot, a price-to-rent ratio and a 0–100 affordability score. It uses the same Bay Area filter, zip code cleaning, `n_homes` and `low_data` rules as Anna's essential-services notebook (`03_anna_essential_services.ipynb`), so the two tables join one-to-one on `zipcode`.

**How to run (Colab):** add the datathon Drive folder to *My Drive* as a shortcut, set `DATA_ROOT` below if your path differs, then `Runtime → Run all`.
**How to run (locally):** put the three Zillow CSVs in `data/raw/zillow housing dataset/` (or set `DATATHON_RAW_DIR`) and run all cells from this folder.

**Outputs:** `albert_listing_by_zip.csv`, `albert_listing_by_county.csv`, 4 chart PNGs, and (if Anna's file is found) `listing_plus_essentials_by_zip.csv`. This is the backup to `04_albert_housing_costs.ipynb`, for when the datathon Zillow files can be read (Drive blocked their download on Nov 14). Its columns are described in the cells below.

## 1. Connect to Google Drive (or use local files)

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/[For Students] 8th Datathon For Social Good"
    OUT_DIR = "/content/drive/MyDrive/datathon_outputs"
except ImportError:                       # running locally, from datathon/albert_upload/notebooks
    DATA_ROOT = os.environ.get("DATATHON_RAW_DIR", "../../../data/raw")
    OUT_DIR = "../outputs"

TABLE_DIR = OUT_DIR if OUT_DIR.endswith("datathon_outputs") else OUT_DIR + "/tables"
FIG_DIR = OUT_DIR if OUT_DIR.endswith("datathon_outputs") else OUT_DIR + "/figures"
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print("Data:", DATA_ROOT, "\nTables:", TABLE_DIR, "\nFigures:", FIG_DIR)

## 2. Load the Zillow files
Only the columns we need are read, in chunks, because the files are 1.5–4 GB each. The `_sample` file in the same Drive folder is skipped: it repeats listings that are already in the three main files.

In [ ]:
import glob
import numpy as np
import pandas as pd

zillow_folder = DATA_ROOT + "/zillow housing dataset"
zillow_files = sorted(f for f in glob.glob(glob.escape(zillow_folder) + "/*.csv")
                      if "sample" not in os.path.basename(f).lower())
print(zillow_files)
if not zillow_files:
    raise FileNotFoundError(
        f"No Zillow CSVs found in {zillow_folder!r}.\n"
        "Colab: check that DATA_ROOT points to the datathon folder in your Drive.\n"
        "Locally: put Zillow_part1.csv, zillow_part2.csv and zillow_part3.csv in that folder "
        "(or set the DATATHON_RAW_DIR environment variable to the folder that contains 'zillow housing dataset').")

WANT = ["zpid", "zipcode", "city", "county", "latitude", "longitude",
        "homeStatus", "homeType", "price", "zestimate", "rentZestimate",
        "bedrooms", "bathrooms", "livingArea"]

parts = []
for f in zillow_files:
    cols = [c for c in WANT if c in pd.read_csv(f, nrows=0).columns]
    for chunk in pd.read_csv(f, usecols=cols, dtype={"zipcode": str}, chunksize=250_000, low_memory=False):
        parts.append(chunk)
raw = pd.concat(parts, ignore_index=True)
print(raw.shape, "rows read")

## 3. Clean and keep the 9 Bay Area counties
Same steps, in the same order, as Anna's notebook: drop rows without coordinates or zip code, keep one row per `zpid`, take the 5-digit zip code, keep the 9 counties. This makes `n_homes` and `low_data` identical in both tables.

In [ ]:
homes = raw.dropna(subset=["latitude", "longitude", "zipcode"]).drop_duplicates("zpid")
homes["zipcode"] = homes["zipcode"].str.extract(r"(\d{5})")[0]
homes = homes.dropna(subset=["zipcode"])
print(homes.shape, "homes in", homes["zipcode"].nunique(), "zip codes (all of CA)")

BAY_AREA = ["Alameda County", "Contra Costa County", "Marin County", "Napa County",
            "San Francisco County", "San Mateo County", "Santa Clara County",
            "Solano County", "Sonoma County"]

homes = homes[homes["county"].isin(BAY_AREA)].reset_index(drop=True)
for c in ["price", "zestimate", "rentZestimate", "bedrooms", "bathrooms", "livingArea"]:
    homes[c] = pd.to_numeric(homes[c], errors="coerce")
homes["homeStatus"] = homes["homeStatus"].astype(str).str.strip().str.upper()
homes["homeType"] = homes["homeType"].astype(str).str.strip().str.upper()

print("Bay Area:", homes.shape, "homes in", homes["zipcode"].nunique(), "zip codes")
print(homes["county"].value_counts().to_string())
print()
print(homes["homeStatus"].value_counts().to_string())
print()
print(homes["homeType"].value_counts().to_string())

## 4. Split listings into rent and home price
`price` means different things by `homeStatus`: monthly rent for `FOR_RENT`, asking price for `FOR_SALE`, sale price for `RECENTLY_SOLD`. So they're split before any median is taken, and impossible values are dropped:

| Measure | Rows used | Kept range |
|---|---|---|
| Listing rent | `FOR_RENT` | $300 – $30,000 / month (same as Anna) |
| Rent Zestimate | every home with one | $300 – $30,000 / month |
| Home value (Zestimate) | every home except `APARTMENT` and `LOT` | $50,000 – $50 million |
| Asking price | `FOR_SALE`, not `LOT` | $50,000 – $50 million |
| Sale price | `RECENTLY_SOLD`, not `LOT` | $50,000 – $50 million |

In [ ]:
RENT_MIN, RENT_MAX = 300, 30_000
PRICE_MIN, PRICE_MAX = 50_000, 50_000_000

rentals = homes[(homes["homeStatus"] == "FOR_RENT") & homes["price"].between(RENT_MIN, RENT_MAX)].copy()

rent_est = homes[homes["rentZestimate"].between(RENT_MIN, RENT_MAX)]

# Zestimate of a whole apartment building (or of land) isn't the price of a home someone could buy
owned = homes[~homes["homeType"].isin(["APARTMENT", "LOT"])]
values = owned[owned["zestimate"].between(PRICE_MIN, PRICE_MAX)].copy()

for_sale = owned[(owned["homeStatus"] == "FOR_SALE") & owned["price"].between(PRICE_MIN, PRICE_MAX)]
sold = owned[(owned["homeStatus"] == "RECENTLY_SOLD") & owned["price"].between(PRICE_MIN, PRICE_MAX)]

# Price per square foot: drop livingArea = 0 or tiny (README data-quality note)
values_sqft = values[values["livingArea"] >= 200]

print(f"Rental listings:          {len(rentals):>7,}  (dropped {((homes['homeStatus']=='FOR_RENT') & ~homes['price'].between(RENT_MIN, RENT_MAX)).sum():,} outside ${RENT_MIN}-${RENT_MAX:,})")
print(f"Homes with rent Zestimate:{len(rent_est):>7,}")
print(f"Homes with a Zestimate:   {len(values):>7,}")
print(f"For-sale listings:        {len(for_sale):>7,}")
print(f"Recently sold:            {len(sold):>7,}")

## 5. Hand-check: 10 random rentals and 10 random homes (seed 1)
Look for rows where the rent looks like a sale price (or the reverse), studios priced like houses, etc. Write what you find in the notes cell below.

In [ ]:
show = ["city", "zipcode", "homeStatus", "homeType", "bedrooms", "livingArea", "price", "rentZestimate", "zestimate"]
print("RENTALS"); print(rentals[show].sample(min(10, len(rentals)), random_state=1).to_string())
print("\nHOMES (Zestimate)"); print(values[show].sample(min(10, len(values)), random_state=1).to_string())
print("\nMost expensive 'rents' kept:"); print(rentals.nlargest(5, "price")[show].to_string())
print("\nCheapest 'home values' kept:"); print(values.nsmallest(5, "zestimate")[show].to_string())

**Hand-check notes:** _(fill in after running: how many of the 20 looked right, and anything odd)_

## 6. Summarize by zip code

In [ ]:
g = homes.groupby("zipcode")
by_zip = pd.DataFrame(index=sorted(homes["zipcode"].unique()))
by_zip.index.name = "zipcode"

# Same identity columns as Anna's table
by_zip["n_homes"] = g.size()
by_zip["city"] = g["city"].agg(lambda s: s.mode().iat[0] if s.notna().any() else None)
by_zip["county"] = g["county"].agg(lambda s: s.mode().iat[0] if s.notna().any() else None)
by_zip["low_data"] = by_zip["n_homes"] < 20
by_zip["latitude"] = g["latitude"].median()        # centre of the zip's homes, for map labels/dots
by_zip["longitude"] = g["longitude"].median()

# Rent
by_zip["typical_rent"] = rentals.groupby("zipcode")["price"].median()
by_zip["n_rentals"] = rentals.groupby("zipcode").size()
for beds, name in [(0, "studio"), (1, "1br"), (2, "2br"), (3, "3br")]:
    sub = rentals[rentals["bedrooms"] == beds]
    med = sub.groupby("zipcode")["price"].median()
    n = sub.groupby("zipcode").size()
    by_zip[f"rent_{name}"] = med.where(n >= 5)          # need 5+ listings of that size
rb = rentals[rentals["bedrooms"] >= 1].assign(rpb=lambda d: d["price"] / d["bedrooms"])
by_zip["rent_per_bedroom"] = rb.groupby("zipcode")["rpb"].median()
by_zip["typical_rent_zestimate"] = rent_est.groupby("zipcode")["rentZestimate"].median()

# Home price
by_zip["typical_home_value"] = values.groupby("zipcode")["zestimate"].median()
by_zip["n_home_values"] = values.groupby("zipcode").size()
by_zip["median_list_price"] = for_sale.groupby("zipcode")["price"].median()
by_zip["n_for_sale"] = for_sale.groupby("zipcode").size()
by_zip["median_sold_price"] = sold.groupby("zipcode")["price"].median()
by_zip["n_sold"] = sold.groupby("zipcode").size()
by_zip["value_per_sqft"] = (values_sqft["zestimate"] / values_sqft["livingArea"]).groupby(values_sqft["zipcode"]).median()

# Years of rent to equal the price of a home (both Zestimates, so the same homes are compared)
by_zip["price_to_rent_ratio"] = by_zip["typical_home_value"] / (12 * by_zip["typical_rent_zestimate"])

for c in ["n_rentals", "n_home_values", "n_for_sale", "n_sold"]:
    by_zip[c] = by_zip[c].fillna(0).astype(int)
by_zip["low_rent_data"] = by_zip["n_rentals"] < 10

print(by_zip.shape)
print("Zips with fewer than 20 homes:", by_zip["low_data"].sum())
print("Zips with 20+ homes:", (~by_zip["low_data"]).sum())
print("Zips with a typical rent:", by_zip["typical_rent"].notna().sum(),
      "| with 10+ rentals:", (~by_zip["low_rent_data"]).sum())
print("Zips with a typical home value:", by_zip["typical_home_value"].notna().sum())

## 7. Affordability score (0–100)
Same method as Anna's access score, so the two can be compared or combined directly. For zip codes that aren't `low_data`, rank them on **rent per bedroom** and on **typical home value** (cheapest ≈ 1, most expensive ≈ 0), average the two ranks and multiply by 100. Rent per bedroom is used instead of plain rent so areas whose listings are mostly studios and in-law units don't look cheaper than they are (see Anna's bias note).

The score is **relative**: 80 means "cheaper than most Bay Area zip codes", not affordable for a given income.

In [ ]:
ok_mask = ~by_zip["low_data"]
score_parts = ["rent_per_bedroom", "typical_home_value"]
ranks = pd.concat([1 - by_zip.loc[ok_mask, c].rank(pct=True) for c in score_parts], axis=1)
by_zip["affordability_score"] = (ranks.mean(axis=1) * 100).round(1)   # NaN for low_data zips

ok = by_zip[ok_mask]
show = ["city", "county", "typical_rent", "rent_per_bedroom", "typical_home_value", "affordability_score", "n_homes"]
print("MOST AFFORDABLE zips"); print(ok.sort_values("affordability_score", ascending=False)[show].head(10).round(0).to_string())
print("\nLEAST AFFORDABLE zips"); print(ok.sort_values("affordability_score")[show].head(10).round(0).to_string())

## 8. Summary by county (all homes in the county, not an average of zip codes)

In [ ]:
county = pd.DataFrame({
    "zip_codes": homes.groupby("county")["zipcode"].nunique(),
    "homes": homes.groupby("county").size(),
    "typical_rent": rentals.groupby("county")["price"].median(),
    "rent_2br": rentals[rentals["bedrooms"] == 2].groupby("county")["price"].median(),
    "rental_listings": rentals.groupby("county").size(),
    "typical_home_value": values.groupby("county")["zestimate"].median(),
    "median_list_price": for_sale.groupby("county")["price"].median(),
    "median_sold_price": sold.groupby("county")["price"].median(),
})
county.index.name = "county"
county = county.sort_values("typical_home_value", ascending=False)
bay = pd.DataFrame({
    "zip_codes": [homes["zipcode"].nunique()], "homes": [len(homes)],
    "typical_rent": [rentals["price"].median()], "rent_2br": [rentals.loc[rentals["bedrooms"] == 2, "price"].median()],
    "rental_listings": [len(rentals)], "typical_home_value": [values["zestimate"].median()],
    "median_list_price": [for_sale["price"].median()], "median_sold_price": [sold["price"].median()],
}, index=pd.Index(["Bay Area (9 counties)"], name="county"))
county = pd.concat([county, bay]).round(0)
county

## 9. Save results

In [ ]:
col_order = ["n_homes", "city", "county", "low_data", "latitude", "longitude",
             "typical_rent", "n_rentals", "low_rent_data", "rent_studio", "rent_1br", "rent_2br", "rent_3br",
             "rent_per_bedroom", "typical_rent_zestimate",
             "typical_home_value", "n_home_values", "median_list_price", "n_for_sale",
             "median_sold_price", "n_sold", "value_per_sqft", "price_to_rent_ratio", "affordability_score"]
by_zip = by_zip[col_order]
by_zip.reset_index().to_csv(TABLE_DIR + "/albert_listing_by_zip.csv", index=False)
county.reset_index().to_csv(TABLE_DIR + "/albert_listing_by_county.csv", index=False)
print("Saved albert_listing_by_zip.csv", by_zip.shape, "and housing_by_county.csv to", TABLE_DIR)

## 10. Join with Anna's essential-services table
If Anna's `essentials_by_zip.csv` is in the outputs folder (Colab) or her `anna_essentials_by_zip.csv` is in the repo (local), this checks that the zip codes line up and writes one combined table for the map. Anna's `median_rent`, `n_rentals` and `rent_per_bedroom` are dropped from the combined table because this notebook's rent columns replace them (same definitions, but duplicate listings removed here).

In [ ]:
anna_paths = [OUT_DIR + "/essentials_by_zip.csv",
              "../../anna_upload/outputs/tables/anna_essentials_by_zip.csv"]
anna_path = next((p for p in anna_paths if os.path.exists(p)), None)

if anna_path is None:
    print("Anna's table not found; skipping the join.")
else:
    anna = pd.read_csv(anna_path, dtype={"zipcode": str})
    mine = by_zip.reset_index()
    both = set(anna["zipcode"]) & set(mine["zipcode"])
    print(f"Anna: {len(anna)} zips | housing: {len(mine)} zips | in both: {len(both)}")
    print("Only in Anna's:", sorted(set(anna["zipcode"]) - set(mine["zipcode"]))[:20])
    print("Only in housing:", sorted(set(mine["zipcode"]) - set(anna["zipcode"]))[:20])
    chk = anna.merge(mine[["zipcode", "n_homes", "low_data"]], on="zipcode", suffixes=("_anna", ""))
    print("n_homes identical in", (chk["n_homes_anna"] == chk["n_homes"]).sum(), "of", len(chk), "zips")

    anna_cols = [c for c in anna.columns if c not in ["n_homes", "city", "county", "low_data",
                                                      "median_rent", "n_rentals", "rent_per_bedroom"]]
    combined = mine.merge(anna[anna_cols], on="zipcode", how="outer", validate="one_to_one")
    combined.to_csv(TABLE_DIR + "/listing_plus_essentials_by_zip.csv", index=False)
    print("Saved listing_plus_essentials_by_zip.csv", combined.shape)

## 11. Charts

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "x", "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlecolor": INK,
})
dollars = FuncFormatter(lambda v, _: f"${v:,.0f}")
millions = FuncFormatter(lambda v, _: f"${v/1e6:.1f}M")
short = lambda s: s.replace(" County", "")

ok = by_zip[~by_zip["low_data"]].copy()
cty = county.drop(index="Bay Area (9 counties)")

# Chart 1: typical rent and home value by county
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)
c1 = cty.sort_values("typical_rent")
bars = axes[0].barh([short(c) for c in c1.index], c1["typical_rent"], color=BLUE, height=0.6)
for bar, v in zip(bars, c1["typical_rent"]):
    axes[0].text(v, bar.get_y() + bar.get_height() / 2, f" ${v:,.0f}", va="center", color=INK2, fontsize=9)
axes[0].set_title("Typical monthly rent")
axes[0].xaxis.set_major_formatter(dollars)
c2 = cty.sort_values("typical_home_value")
bars = axes[1].barh([short(c) for c in c2.index], c2["typical_home_value"], color=BLUE, height=0.6)
for bar, v in zip(bars, c2["typical_home_value"]):
    axes[1].text(v, bar.get_y() + bar.get_height() / 2, f" ${v/1e6:.2f}M", va="center", color=INK2, fontsize=9)
axes[1].set_title("Typical home value")
axes[1].xaxis.set_major_formatter(millions)
for ax in axes:
    ax.tick_params(axis="y", length=0)
    ax.set_xlim(0, ax.get_xlim()[1] * 1.18)
fig.suptitle("What housing costs in each Bay Area county (median of Zillow listings)",
             x=0.01, ha="left", fontweight="bold", color=INK)
plt.tight_layout(); plt.savefig(FIG_DIR + "/albert_listing_cost_by_county.png", dpi=200); plt.show()

# Chart 2: the 10 most and 10 least expensive zip codes by rent per bedroom
rel = ok[~ok["low_rent_data"]].dropna(subset=["rent_per_bedroom"])
low = rel.nsmallest(10, "rent_per_bedroom")
high = rel.nlargest(10, "rent_per_bedroom").iloc[::-1]
both = pd.concat([low.iloc[::-1], high])
labels = [f"{z}  {c}" for z, c in zip(both.index, both["city"])]
colors = [BLUE] * len(low) + [ORANGE] * len(high)
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(labels, both["rent_per_bedroom"], color=colors, height=0.65)
for bar, v in zip(bars, both["rent_per_bedroom"]):
    ax.text(v, bar.get_y() + bar.get_height() / 2, f" ${v:,.0f}", va="center", color=INK2, fontsize=10)
ax.axhline(len(low) - 0.5, color=GRID, linewidth=1)
ax.set_xlim(0, both["rent_per_bedroom"].max() * 1.15)
ax.xaxis.set_major_formatter(dollars)
ax.set_xlabel("Median monthly rent per bedroom (zip codes with 10+ rental listings)")
ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(color=BLUE, label="Cheapest"), Patch(color=ORANGE, label="Most expensive")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.set_title("The 10 cheapest and 10 most expensive zip codes to rent in", pad=28)
plt.tight_layout(); plt.savefig(FIG_DIR + "/albert_listing_rent_ranking.png", dpi=200); plt.show()

# Chart 3: the 10 cheapest and 10 most expensive zip codes by home value
rel = ok[ok["n_home_values"] >= 20].dropna(subset=["typical_home_value"])
low = rel.nsmallest(10, "typical_home_value")
high = rel.nlargest(10, "typical_home_value").iloc[::-1]
both = pd.concat([low.iloc[::-1], high])
labels = [f"{z}  {c}" for z, c in zip(both.index, both["city"])]
fig, ax = plt.subplots(figsize=(8, 7))
bars = ax.barh(labels, both["typical_home_value"], color=colors, height=0.65)
for bar, v in zip(bars, both["typical_home_value"]):
    ax.text(v, bar.get_y() + bar.get_height() / 2, f" ${v/1e6:.2f}M", va="center", color=INK2, fontsize=10)
ax.axhline(len(low) - 0.5, color=GRID, linewidth=1)
ax.set_xlim(0, both["typical_home_value"].max() * 1.15)
ax.xaxis.set_major_formatter(millions)
ax.set_xlabel("Median Zestimate (zip codes with 20+ valued homes)")
ax.tick_params(axis="y", length=0)
ax.legend(handles=[Patch(color=BLUE, label="Cheapest"), Patch(color=ORANGE, label="Most expensive")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
ax.set_title("The 10 cheapest and 10 most expensive zip codes to buy in", pad=28)
plt.tight_layout(); plt.savefig(FIG_DIR + "/albert_listing_home_value_ranking.png", dpi=200); plt.show()

# Chart 4: rent vs home value per zip code
rel = ok.dropna(subset=["typical_rent_zestimate", "typical_home_value"])
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.grid(axis="both")
ax.scatter(rel["typical_rent_zestimate"], rel["typical_home_value"], s=36, color=BLUE, alpha=0.75,
           edgecolor=SURFACE, linewidth=1)
xs = np.linspace(rel["typical_rent_zestimate"].min(), rel["typical_rent_zestimate"].max(), 50)
med_ratio = rel["price_to_rent_ratio"].median()
ax.plot(xs, xs * 12 * med_ratio, color=MUTED, linewidth=1, linestyle="--")
ax.text(xs[-1], xs[-1] * 12 * med_ratio, f"  median: {med_ratio:.0f} years of rent", color=MUTED,
        fontsize=9, va="center", ha="right")
for (z, r), dy in zip(rel.nlargest(2, "price_to_rent_ratio").iterrows(), [14, -8]):
    ax.annotate(f"{z} {r['city']}", (r["typical_rent_zestimate"], r["typical_home_value"]), xytext=(-14, dy),
                textcoords="offset points", fontsize=9, color=INK, ha="right",
                arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.8))
for (z, r), dy in zip(rel.nsmallest(2, "price_to_rent_ratio").iterrows(), [-16, 6]):
    ax.annotate(f"{z} {r['city']}", (r["typical_rent_zestimate"], r["typical_home_value"]), xytext=(14, dy),
                textcoords="offset points", fontsize=9, color=INK,
                arrowprops=dict(arrowstyle="-", color=MUTED, linewidth=0.8))
ax.xaxis.set_major_formatter(dollars)
ax.yaxis.set_major_formatter(millions)
ax.set_xlabel("Typical rent (median rent Zestimate, $/month)")
ax.set_ylabel("Typical home value (median Zestimate)")
ax.set_title("Each dot is a zip code: above the line, buying costs more years of rent")
plt.tight_layout(); plt.savefig(FIG_DIR + "/albert_listing_rent_vs_value.png", dpi=200); plt.show()

## 12. Numbers for the write-up

In [ ]:
ok = by_zip[~by_zip["low_data"]]
print(f"Zip codes: {len(by_zip)} total, {len(ok)} with 20+ homes")
print(f"Bay Area typical rent (all listings): ${rentals['price'].median():,.0f}/month from {len(rentals):,} listings")
print(f"Bay Area typical home value: ${values['zestimate'].median():,.0f} from {len(values):,} homes")
print(f"Typical rent across zips ranges ${ok['typical_rent'].min():,.0f} to ${ok['typical_rent'].max():,.0f}")
print(f"Typical home value across zips ranges ${ok['typical_home_value'].min():,.0f} to ${ok['typical_home_value'].max():,.0f}")
print(f"Median price-to-rent ratio: {ok['price_to_rent_ratio'].median():.1f} years")
print(f"Listing rent vs rent Zestimate, median gap across zips: "
      f"{((ok['typical_rent'] - ok['typical_rent_zestimate']) / ok['typical_rent_zestimate']).median()*100:+.1f}%")